In [1]:
# Week 8 Day 3 Daily Challenge:
# Hugging Face Introduction Paragraph

# In this exercise, we will use Hugging Face Transformers, a popular Python library that provides easy access to pre-trained models for natural language processing. Hugging Face allows us to quickly load models like BERT or GPT and use them without training from scratch.

In [2]:
!pip install transformers

In [3]:
from huggingface_hub import login
from google.colab import userdata

# Get your token from Colab secrets
hf_token = userdata.get('HUGGINGFACE_KEY') # Assuming you named your secret 'HUGGINGFACE_KEY'

# Login to Hugging Face Hub
login(token=hf_token)

print("Successfully logged in to Hugging Face Hub.")

Successfully logged in to Hugging Face Hub.


In [4]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
inputs = tokenizer("Hello Hugging Face!", return_tensors="pt")
outputs = model(**inputs)

In [6]:
# Daily Challenge: Build a Retrieval Augmented Generation (RAG) System


# 👩‍🏫 👩🏿‍🏫 What You’ll learn

#     Implement a Retrieval Augmented Generation (RAG) system using Langchain and Hugging Face.
#     Load and process datasets using Hugging Face datasets and Langchain HuggingFaceDatasetLoader.
#     Split documents into smaller chunks using Langchain RecursiveCharacterTextSplitter.
#     Generate text embeddings using Hugging Face sentence-transformers and Langchain HuggingFaceEmbeddings.
#     Create and utilize vector stores with Langchain FAISS for efficient document retrieval.
#     Prepare and integrate a pre-trained Language Model (LLM) from Hugging Face transformers for question answering.
#     Build a Retrieval QA Chain using Langchain RetrievalQA to answer questions based on retrieved documents.


# 🛠️ What you will create

# You will create a functional RAG system that can answer questions based on a dataset loaded from Hugging Face Datasets. This system will:

#     Load the databricks/databricks-dolly-15k dataset.
#     Index the dataset content into a vector store.
#     Utilize a pre-trained question-answering model from Hugging Face.
#     Answer user queries by retrieving relevant documents and using the LLM to generate answers.
# Task

# Our task is to implement RAG using Langchain and Hugging Face!

# 1. Set up your environment: : This ensures all the necessary tools are available to build the RAG system. Each library serves a specific role: Langchain handles the orchestration of components, transformers provide pre-trained models, sentence-transformers generate embeddings, datasets load sample data, and FAISS enables fast similarity searches.

#     Open your terminal or notebook environment.
#     Install all required libraries by running these commands:



In [7]:
!pip install -q langchain
!pip install -q torch
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -q datasets
!pip install -q faiss-cpu
!pip install -U langchain-community

In [8]:
# 2. Load the dataset: To provide the system with information to retrieve from, you’ll load a real-world dataset. HuggingFaceDatasetLoader simplifies the process of accessing Hugging Face datasets and formatting them into documents that Langchain can process.

#     before loading the dataset, run :

!pip install -Uq datasets


In [9]:
# Reinstall pyarrow and datasets to fix binary incompatibility issue
!pip uninstall -y pyarrow datasets
!pip install -Uq pyarrow
!pip install -Uq datasets

Found existing installation: pyarrow 24.0.0
Uninstalling pyarrow-24.0.0:
  Successfully uninstalled pyarrow-24.0.0
Found existing installation: datasets 4.8.5
Uninstalling datasets-4.8.5:
  Successfully uninstalled datasets-4.8.5


In [10]:
from langchain_community.document_loaders import HuggingFaceDatasetLoader
dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

In [11]:
#     Create a HuggingFaceDatasetLoader instance and load the data as documents:


loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()
print(data[:2]) # Optional: Print the first 2 entries to verify loading


[Document(metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}, page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia\'s domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."'), Document(metadata={'instruction': 'Which is a species of fish? Tope or Rope', 'response': 'Tope', 'category': 'classification'}, page_content='""')]


In [12]:
# 3. Split the documents: Language models have a limit on how much text they can process at once. Splitting large documents into smaller, overlapping chunks ensures that no important context is lost and that each piece of text is a manageable size for embedding and retrieval.

from langchain_text_splitters import RecursiveCharacterTextSplitter
#     Create a RecursiveCharacterTextSplitter instance with a chunk_size of 1000 and chunk_overlap of 150:


text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

In [13]:
#     Split the loaded documents:


docs = text_splitter.split_documents(data)
print(docs[0]) # Optional: Print the first document chunk


page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."' metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [14]:
# 4. Embed the text: Text needs to be converted into numerical representations (embeddings) so that similar pieces of text can be found efficiently. Using a sentence-transformer model creates embeddings that capture semantic meaning, enabling effective retrieval later.

#     Install the new langchain-huggingface package if not already installed
!pip install -U langchain-huggingface

#     Import HuggingFaceEmbeddings from langchain.embeddings.
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
#     Define the model path, model configurations, and encoding options:


modelPath = "sentence-transformers/all-MiniLM-l6-v2"
model_kwargs = {'device':'cpu'}
encode_kwargs = {'normalize_embeddings': False}


#     Initialize HuggingFaceEmbeddings:


embeddings = HuggingFaceEmbeddings(
  model_name=modelPath,
  model_kwargs=model_kwargs,
  encode_kwargs=encode_kwargs
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-l6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
#     (Optional) Test embedding creation:


text = "This is a test document."
query_result = embeddings.embed_query(text)
print(query_result[:3])

[-0.038338519632816315, 0.1234646886587143, -0.028642984107136726]


In [16]:
# 5. Create a vector store: A vector store like FAISS indexes the embeddings, allowing fast and scalable similarity searches. This is how the system quickly finds relevant pieces of text when a query is made.

#     Import FAISS from langchain.vectorstores.
from langchain_community.vectorstores import FAISS
#     Create a FAISS vector store from the document chunks and embeddings:

db = FAISS.from_documents(docs, embeddings)


#     Note: This step might take some time depending on your dataset size.


In [17]:
# 6. Prepare the LLM model: The Language Model is responsible for generating answers based on retrieved documents. Loading a pre-trained model and wrapping it in a Langchain pipeline makes it easy to integrate with the retrieval system.

#     Import necessary classes from transformers and langchain:


from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain_huggingface import HuggingFacePipeline


In [23]:
#     Load the tokenizer and question-answering model:

tokenizer = AutoTokenizer.from_pretrained("Intel/dynamic_tinybert")
model = AutoModelForQuestionAnswering.from_pretrained("Intel/dynamic_tinybert")



Invalid model-index. Not loading eval results into CardData.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: Intel/dynamic_tinybert
Key                          | Status     |  | 
-----------------------------+------------+--+-
fit_dense.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
fit_dense.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [24]:
#     Create a question-answering pipeline:


model_name = "Intel/dynamic_tinybert"
tokenizer = AutoTokenizer.from_pretrained(model_name, padding=True, truncation=True, max_length=512)
Youtubeer = pipeline(
  "question-answering",
  model=model_name,
  tokenizer=tokenizer,
  return_tensors='pt'
)



Invalid model-index. Not loading eval results into CardData.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: Intel/dynamic_tinybert
Key                          | Status     |  | 
-----------------------------+------------+--+-
fit_dense.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
fit_dense.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [57]:
from langchain_core.outputs import Generation, LLMResult
from langchain_core.language_models.llms import BaseLLM
from typing import Any, List, Optional

class CustomQALLM(BaseLLM):
    qa_pipeline: Any

    def __init__(self, qa_pipeline: Any, **kwargs: Any):
        super().__init__(qa_pipeline=qa_pipeline, **kwargs)

    def _call(self, prompt: str, stop: Optional[List[str]] = None, **kwargs: Any) -> str:
        # Find the question marker. It's typically the last 'Question:' in the prompt.
        question_start_idx = prompt.rfind('Question:')

        if question_start_idx == -1:
            # Fallback if 'Question:' marker is not found (unlikely with RetrievalQA)
            print("Warning: 'Question:' marker not found in prompt. Using entire prompt as question.")
            extracted_question = prompt.strip()
            extracted_context = "No specific context provided."
        else:
            # Extract the raw question segment after 'Question:'
            raw_question_segment = prompt[question_start_idx + len('Question:'):].strip()

            # 'Helpful Answer:' often follows the actual question in the prompt template
            helpful_answer_idx = raw_question_segment.find('Helpful Answer:')
            if helpful_answer_idx != -1:
                extracted_question = raw_question_segment[:helpful_answer_idx].strip()
            else:
                extracted_question = raw_question_segment.strip()

            # The context is everything before the 'Question:' marker
            context_segment = prompt[:question_start_idx].strip()

            # The prompt preamble typically ends with 'answer.\n'
            instruction_end_marker = 'answer.\n'
            if instruction_end_marker in context_segment:
                extracted_context = context_segment.split(instruction_end_marker, 1)[1].strip()
            else:
                extracted_context = context_segment.strip() # Fallback, might include some instruction text

        if not extracted_question:
            print("Warning: Final extracted question is empty. Using generic fallback.")
            extracted_question = "What is the answer?"

        if not extracted_context:
            print("Warning: Final extracted context is empty. Using entire original prompt as context for QA pipeline.")
            extracted_context = prompt # Fallback for context

        # Call the transformers QA pipeline with the extracted question and context
        qa_result = self.qa_pipeline(question=extracted_question, context=extracted_context)

        # The transformers QA pipeline typically returns a dictionary directly with the answer.
        if qa_result and isinstance(qa_result, dict) and "answer" in qa_result:
            return qa_result["answer"]
        else:
            print(f"Warning: QA pipeline returned an unexpected result: {qa_result}. Returning 'No answer found.'")
            return "No answer found."

    def _generate(self, prompts: List[str], stop: Optional[List[str]] = None, **kwargs: Any) -> LLMResult:
        generations = []
        for prompt in prompts:
            # Call the _call method to get the answer
            answer = self._call(prompt, stop=stop, **kwargs)
            generations.append([Generation(text=answer)])
        return LLMResult(generations=generations)

    @property
    def _llm_type(self) -> str:
        return "custom_qa_llm"

llm = CustomQALLM(Youtubeer)

In [31]:
# 7. Build the Retrieval QA Chain: The Retrieval QA Chain connects the retriever (which finds relevant documents) with the LLM (which generates answers). This chain enables the full RAG process, where the system retrieves helpful context and then answers the user’s query based on that context.

    # Ensure langchain is up-to-date to resolve potential missing modules
!pip install -U langchain
!pip install langchain-classic

In [61]:
# from langchain.chains import RetrievalQA
from langchain_classic.chains import RetrievalQA
# Create a retriever from your FAISS database:


retriever = db.as_retriever(search_kwargs={"k": 4}) # Optional: You can adjust k for number of documents retrieved

In [64]:
#     Build the RetrievalQA chain:


qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever, return_source_documents=True)

In [66]:
question = "What is cheesemaking?"

# Run the QA chain and print the result:
result_dict = qa.invoke({"query": question})
print("Answer:", result_dict["result"])
print("Source Documents:", result_dict["source_documents"])

Answer: to control the spoiling of milk into cheese
Source Documents: [Document(id='49c8799a-64d3-442b-bc36-f14385d0f05a', metadata={'instruction': 'From the provided description of the cheesemaking process, list the ingredients required to make cheese.', 'response': "The main ingredient used to make cheese is milk. Cow's milk is commonly used, though goat, sheep or buffalo can also be used, as could the milk of any mammal in theory. Starter cultures are typically added to aid in the culturing stage of the process. Rennet is added to the cheese milk to promote the separation into cheese curd and whey. Ultimately, salt is added to halt the production of acid later in the process. Finally, mould spores are introduced to assist in ripening. This can be added to the cheese milk early on in the process, or just prior to maturing.", 'category': 'information_extraction'}, page_content='"The goal of cheese making is to control the spoiling of milk into cheese. The milk is traditionally from a 